In [1]:
#basic conversation with gemini using api
from dotenv import load_dotenv
import os 

load_dotenv()
import google.generativeai as genai

# Access your API key
api_key = os.getenv("GEMINI_API_KEY")
#print(api_key)

genai.configure(api_key=api_key)

model = genai.GenerativeModel("gemini-3-flash-preview")

chat = model.start_chat()

response = chat.send_message("Hi, what is machine learning?")
print(response.text)

response = chat.send_message("Explain it in one line")
print(response.text)

C:\Users\varni\.conda\envs\varnit\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
C:\Users\varni\.conda\envs\varnit\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\varni\AppData\Local\Temp\ipykernel_11764\1880783292.py:6: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

ht

At its simplest, **Machine Learning (ML)** is a branch of Artificial Intelligence (AI) that allows computers to learn and make decisions without being explicitly programmed for every specific task.

Instead of a human writing a long list of instructions (rules), the computer uses **data** and **algorithms** to find patterns and teach itself how to perform a task.

---

### How is it different from traditional programming?

*   **Traditional Programming:** A human writes the rules.
    *   *Example:* "If an email contains the word 'Winner' and 'Cash,' move it to the Spam folder."
*   **Machine Learning:** The human gives the computer thousands of examples of spam and non-spam emails. The computer looks at the data and figures out its own rules for identifying spam.

---

### How does it work? (The 3-Step Process)

1.  **Data Input:** You feed the computer a large amount of data (images, numbers, or text).
2.  **Training:** The machine uses an "algorithm" (a mathematical formula) to look

In [2]:
from langchain_community.embeddings import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text")

C:\Users\varni\AppData\Local\Temp\ipykernel_11764\2424380868.py:3: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text")


In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

In [4]:
loader = PyPDFLoader("CancerData.pdf")  
docs = loader.load()

print("Pages loaded:", len(docs))

Pages loaded: 3


In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

all_splits = text_splitter.split_documents(docs)

print("Chunks created:", len(all_splits))

Chunks created: 11


In [13]:
embeddings = OllamaEmbeddings(model="nomic-embed-text") #smarter to understand, better but slow

In [14]:
vector_store = FAISS.from_documents(all_splits, embeddings)

print("Vector store ready ✅")

Vector store ready ✅


In [15]:
query = "What is Immunotherapy?"

results = vector_store.similarity_search(query, k=2)

for doc in results:
    print(doc.page_content)
    print("----")

What  is  Cancer?  
Cancer  is  a  group  of  diseases  in  which  some  of  the  body’s  cells  grow  uncontrollably  and  
spread
 
to
 
other
 
parts
 
of
 
the
 
body.
 
Normally,
 
cells
 
grow,
 
divide,
 
and
 
die
 
in
 
an
 
orderly
 
way.
 
In
 
cancer,
 
this
 
control
 
system
 
breaks
 
down.
 
How  Cancer  Develops  
Cancer  begins  at  the  cell  level  due  to  changes  (mutations)  in  DNA.
----
Treatment  Options  
Treatment  depends  on  cancer  type,  stage,  and  patient  health:  
1.  Surgery  –  removes  tumor  2.  Chemotherapy  –  uses  drugs  to  kill  cancer  cells  3.  Radiation  therapy  –  uses  high-energy  rays  4.  Immunotherapy  –  boosts  immune  system  to  fight  cancer  5.  Targeted  therapy  –  attacks  specific  cancer  cells  6.  Hormone  therapy  –  blocks  hormones  that  fuel  cancer  
Often,  a  combination  of  treatments  is  used.  
What  is  Metastasis?
----


In [16]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.chat_models import ChatOllama

In [17]:
llm = ChatOllama(model="mistral") #best to answer, faster than nomic-embed-text

In [19]:
query = "What is Immunotherapy?"

# Retrieval
docs = vector_store.similarity_search(query, k=2)

context = "\n\n".join(doc.page_content for doc in docs)

# Generation
prompt = f"""
Answer the question using the context below only if you dont know return i dont know.

Context:
{context}

Question:
{query}
"""

response = llm.invoke(prompt)

print(response.content)

 Immunotherapy is one of the treatment options for cancer. It works by boosting the body's own immune system to fight cancer. This type of treatment helps your immune system identify and attack the cancer cells more effectively.
